# M1-01 — MLP cơ bản

Code tương ứng với `m1_01.py`. Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation và test.

Train tối đa 50 epoch, dừng khi validation loss không giảm trong 8 epoch liên tiếp. Test dùng trọng số có validation loss thấp nhất giữ trong RAM. Notebook chưa được chạy.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

from metrics import calculate_metrics




In [2]:
# 1. Cấu hình
DATA_DIR = "../../data/VN_trash_classification_preprocessing"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)




Device: cuda


In [3]:
# 2. Xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])



In [4]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR + "/train", transform=transform)
val_dataset = ImageFolder(DATA_DIR + "/val", transform=transform)
test_dataset = ImageFolder(DATA_DIR + "/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))




Train: 9542
Validation: 1685
Test: 864


In [5]:
# 4. Tạo MLP
class MLP(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 128 * 128, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, images):
        return self.layers(images)


model = MLP(len(train_dataset.classes)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




In [ ]:
# 5. Train và Validation
best_val_loss = float("inf")
best_weights = None
best_epoch = 0
patience = 15
epochs_without_improvement = 0

for epoch in range(50):
    model.train()
    train_loss = 0
    train_true_labels = []
    train_predicted_labels = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        train_true_labels.extend(labels.cpu().tolist())
        train_predicted_labels.extend(predictions.cpu().tolist())

    train_loss /= len(train_dataset)
    train_metrics = calculate_metrics(train_true_labels, train_predicted_labels)
    train_accuracy = train_metrics["accuracy"]

    model.eval()
    val_loss = 0
    val_true_labels = []
    val_predicted_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            val_true_labels.extend(labels.cpu().tolist())
            val_predicted_labels.extend(predictions.cpu().tolist())

    val_loss /= len(val_dataset)
    val_metrics = calculate_metrics(val_true_labels, val_predicted_labels)
    val_accuracy = val_metrics["accuracy"]

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/50 | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}"
    )

    # Dừng khi validation loss không giảm trong 15 epoch liên tiếp.
    if epochs_without_improvement >= patience:
        print(f"Early stopping tại epoch {epoch + 1}")
        break




Epoch 1/50 | Train Loss: 2.8054 | Train Acc: 11.42% | Val Loss: 2.1975 | Val Acc: 11.75%
Epoch 2/50 | Train Loss: 2.1984 | Train Acc: 11.76% | Val Loss: 2.1960 | Val Acc: 11.69%
Epoch 3/50 | Train Loss: 2.1976 | Train Acc: 11.95% | Val Loss: 2.1855 | Val Acc: 11.75%
Epoch 4/50 | Train Loss: 2.1937 | Train Acc: 11.75% | Val Loss: 2.1899 | Val Acc: 11.75%
Epoch 5/50 | Train Loss: 2.1903 | Train Acc: 11.71% | Val Loss: 2.1833 | Val Acc: 11.75%
Epoch 6/50 | Train Loss: 2.1874 | Train Acc: 12.51% | Val Loss: 2.1802 | Val Acc: 12.88%
Epoch 7/50 | Train Loss: 2.1871 | Train Acc: 12.60% | Val Loss: 2.1844 | Val Acc: 12.23%
Epoch 8/50 | Train Loss: 2.1869 | Train Acc: 12.15% | Val Loss: 2.1820 | Val Acc: 13.95%
Epoch 9/50 | Train Loss: 2.1870 | Train Acc: 12.46% | Val Loss: 2.1787 | Val Acc: 13.06%
Epoch 10/50 | Train Loss: 2.1824 | Train Acc: 12.94% | Val Loss: 2.1762 | Val Acc: 13.29%
Epoch 11/50 | Train Loss: 2.1814 | Train Acc: 13.54% | Val Loss: 2.1749 | Val Acc: 13.29%
Epoch 12/50 | Train

In [7]:
# 6. Test
model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.6f}")
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print("\nTest Result")
print(f"Test Accuracy: {metrics['accuracy']:.2%}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall: {metrics['macro_recall']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")


Best epoch: 48 | Best Val Loss: 2.045334

Test Result
Test Accuracy: 19.91%
Macro Precision: 0.1863
Macro Recall: 0.1925
Macro F1: 0.1604
Weighted F1: 0.1564
